In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import re
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from wordcloud import WordCloud
from sklearn.feature_extraction.text import CountVectorizer
from scipy.sparse import hstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split, cross_val_score
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay,
                             classification_report, accuracy_score,
                             precision_score, recall_score, f1_score, roc_auc_score)
from sklearn.feature_selection import SelectKBest, chi2
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline

In [ ]:
# Load dataset
df = pd.read_csv("/content/drive/MyDrive/base_tjsp.csv")

# Dataset dimensions
shape_data = df.shape
print(f"The dataset has {shape_data[0]} rows and {shape_data[1]} attributes.")
df.head(3)

In [ ]:
# Class distribution: 'Cybercrime' vs 'Other'
class_distribution = df['classificacao_level0'].value_counts(normalize=True)

for label, proportion in class_distribution.items():
    print(f"Class {label}: {proportion * 100:.2f}%")

In [ ]:
# Concatenate ruling summary (ementa) and full ruling text (julgado)
df_model = df.copy()
df_model["full_text"] = df["ementa"] + " " + df["julgado"]

In [ ]:
# Extract judgment year from date column
df_model['judgment_year'] = df_model['data_julgamento'].str.extract(r'(\d{4})')

# Assign time period based on judgment year
conditions = [
    df_model['judgment_year'].isin(['2010', '2011', '2012', '2013', '2014', '2015']),
    df_model['judgment_year'].isin(['2016', '2017', '2018', '2019', '2020']),
    df_model['judgment_year'].isin(['2021', '2022', '2023', '2024'])
]
choices = ['2010-2015', '2016-2020', 'after 2020']
df_model['period'] = np.select(conditions, choices, default='before 2010')

In [ ]:
lemmatizer = nltk.stem.WordNetLemmatizer()
stopwords_pt = set(stopwords.words('portuguese'))

# Build pattern lists for rapporteur names, cities, and adjudicating bodies to be removed from text
rapporteur_names = "|".join(df_model['relator'].dropna().str.lower().unique())
city_names       = "|".join(df_model['comarca'].dropna().str.lower().unique())
body_names       = "|".join(df_model['orgao_julgador'].dropna().str.lower().unique())


# Text preprocessing function
def preprocess_text(text):
    if pd.isnull(text):
        return ""

    text = text.lower()
    text = re.sub(r".*?(acórdão)", " ", text)

    # Remove headers, signatures, names, dates, and other patterns
    substitutions = [
        r"poder judiciário\s+tribunal de justiça (do estado de são paulo|do estado)?",
        r"tribunal de justiça\s+poder judiciário",
        r"assinatura eletrônica",
        r"\n",
        rapporteur_names,
        city_names,
        body_names,
        r"\b\d{1,2}[-/.]\d{1,2}[-/.]\d{2,4}\b",
        r"[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.(com|com\.br)\b",
        r"https?://\S+|www\.\S+",
        r"[ºª°]",
        r"[0-9]",
        r"\b[A-Za-z]\b",
    ]

    for pattern in substitutions:
        text = re.sub(pattern, " ", text)

    # Remove punctuation and extra whitespace
    text = text.translate(str.maketrans("", "", string.punctuation))
    text = re.sub(r"\s+", " ", text).strip()

    # Tokenize
    text = word_tokenize(text)

    # Remove stopwords
    text = [word for word in text if word not in stopwords_pt]

    # Lemmatize
    text = [lemmatizer.lemmatize(word) for word in text]

    return text

# Apply preprocessing
df_model['processed_text'] = df_model['full_text'].apply(preprocess_text)

In [ ]:
# Represent text using TF-IDF
tfidf = TfidfVectorizer()
X_tfidf = tfidf.fit_transform(df_model['processed_text'].apply(lambda x: ' '.join(x)))

In [ ]:
# One-Hot Encoding for categorical columns
encoder = OneHotEncoder(handle_unknown='ignore')
X_cat = encoder.fit_transform(df_model[['classe', 'assunto', 'topic', 'period']])

# Combine TF-IDF and One-Hot Encoding features
X_combined = hstack([X_tfidf, X_cat])

# Define features and target
X = X_combined
y = df_model['classificacao_level0']

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Combined feature matrix shape
print(f"Combined feature matrix shape: {X_combined.shape}")

In [ ]:
# Stratified cross-validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# Candidate values for the number of selected features
k_values = [100, 300, 500, 1000, 2000, 3000, 5000, 7000]

SVM

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('svm', SVC(kernel='linear', probability=True, random_state=42))
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score  = np.std(f1_scores)

    results.append({
        "name": f"SVM with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('svm', SVC(kernel='linear', probability=True, random_state=42))
])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: SVM with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['svm'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - SVM with k={best_k}")
plt.show()

Logistic Regression

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42))
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score = np.std(f1_scores)

    results.append({
        "name": f"Logistic Regression with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('LogisticRegression', LogisticRegression(max_iter=1000, random_state=42))
])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: Logistic Regression with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['LogisticRegression'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - Logistic Regression with k={best_k}")
plt.show()

Naive Bayes

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('MultinomialNB', MultinomialNB())
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score = np.std(f1_scores)

    results.append({
        "name": f"Naive Bayes with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('MultinomialNB', MultinomialNB())
])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: Naive Bayes with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['MultinomialNB'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - Naive Bayes with k={best_k}")
plt.show()

Random Forest

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42))
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score = np.std(f1_scores)

    results.append({
        "name": f"Random Forest with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('RandomForest', RandomForestClassifier(n_estimators=100, random_state=42))
  ])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: Random Forest with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['RandomForest'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - Random Forest with k={best_k}")
plt.show()

XGBoost

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('XGBoost', XGBClassifier(eval_metric='logloss', random_state=42))
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score = np.std(f1_scores)

    results.append({
        "name": f"XGBoost with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('XGBoost', XGBClassifier(eval_metric='logloss', random_state=42))
  ])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: XGBoost with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['XGBoost'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - XGBoost with k={best_k}")
plt.show()

LGBM

In [ ]:
results = []

for k in k_values:
    pipe = Pipeline([
        ('select', SelectKBest(score_func=chi2, k=k)),
        ('LightGBM', lgb.LGBMClassifier(random_state=42, verbose=-1))
    ])

    # Cross-validated F1 score
    f1_scores = cross_val_score(pipe, X, y, cv=cv, scoring='f1')
    mean_score = np.mean(f1_scores)
    std_score = np.std(f1_scores)

    results.append({
        "name": f"LGBM with k={k}",
        "k": k,
        "f1_mean": mean_score,
        "f1_std": std_score,
        "scores": f1_scores
    })

    print(f"k={k} → Mean F1: {mean_score:.4f} | Std: {std_score:.4f}")

# Identify the best k based on highest mean F1
best = max(results, key=lambda x: x["f1_mean"])
print(f"\n Best model: {best['name']} → Mean F1: {best['f1_mean']:.4f}")

# F1 score vs k curve
plt.figure(figsize=(8, 4))
plt.plot([r['k'] for r in results], [r['f1_mean'] for r in results], marker='o', label='Mean F1', color='purple')
plt.fill_between(
    [r['k'] for r in results],
    [r['f1_mean'] - r['f1_std'] for r in results],
    [r['f1_mean'] + r['f1_std'] for r in results],
    color='purple', alpha=0.2, label='±1 Std Dev'
)
plt.title("Mean F1 Score vs Number of Selected Features (k)")
plt.xlabel("Number of Features (k)")
plt.ylabel("Mean F1 Score")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
best_k = best['k']

# Train-test split for final evaluation
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# Build and train the final pipeline with the best k
pipe_final = Pipeline([
    ('select', SelectKBest(score_func=chi2, k=best_k)),
    ('lightgbm', lgb.LGBMClassifier(random_state=42, verbose=-1))
  ])
pipe_final.fit(X_train_final, y_train_final)

# Predictions
y_pred_final = pipe_final.predict(X_test_final)
y_prob_final = pipe_final.predict_proba(X_test_final)[:, 1]

# Evaluation metrics
print(f"\n Final Evaluation - Best Model: LightGBM with k={best_k}")
print(classification_report(y_test_final, y_pred_final))
print("Accuracy:",      accuracy_score(y_test_final, y_pred_final))
print("Precision:",     precision_score(y_test_final, y_pred_final))
print("Recall:",        recall_score(y_test_final, y_pred_final))
print("F1 Score:",      f1_score(y_test_final, y_pred_final))
print("ROC AUC Score:", roc_auc_score(y_test_final, y_prob_final))

# Confusion matrix
cm = confusion_matrix(y_test_final, y_pred_final, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=pipe_final.named_steps['lightgbm'].classes_)
disp.plot(cmap="PuRd", values_format=".1%")
plt.title(f"Confusion Matrix - LightGBM with k={best_k}")
plt.show()